# Unidad 3 – Sesión 2
## Transformación de variables para análisis y modelado

**Curso:** Minería de Datos  
**Fase CRISP-DM:** Preparación de los Datos

### Propósito
- corregir tipos de datos;
- transformar fechas;
- crear variables derivadas;
- codificar variables categóricas;
- escalar variables numéricas;
- validar las transformaciones.

> **Idea clave:** Limpiar los datos no es suficiente; también debemos representarlos de una forma útil para el análisis.

In [1]:
import pandas as pd
import numpy as np

data = {
    "cliente_id": [101, 102, 103, 104, 105],
    "edad": ["25", "32", "40", "29", "36"],
    "ciudad": ["LIMA", "CUSCO", "LIMA", "AREQUIPA", "CUSCO"],
    "genero": ["M", "F", "F", "M", "F"],
    "fecha_registro": ["2026-01-10","2026-02-15","2026-02-20","2026-03-05","2026-03-21"],
    "ingreso": [1500, 3200, 2100, 1800, 4500],
    "activo": ["SI", "NO", "SI", "SI", "NO"]
}

df = pd.DataFrame(data)
df

,cliente_id,edad,ciudad,genero,fecha_registro,ingreso,activo
0,101,25,LIMA,M,2026-01-10,1500,SI
1,102,32,CUSCO,F,2026-02-15,3200,NO
2,103,40,LIMA,F,2026-02-20,2100,SI
3,104,29,AREQUIPA,M,2026-03-05,1800,SI
4,105,36,CUSCO,F,2026-03-21,4500,NO


# 1. Revisar tipos de datos

In [2]:
df.dtypes

,0
cliente_id,int64
edad,object
ciudad,object
genero,object
fecha_registro,object
ingreso,int64
activo,object


`edad` es conceptualmente numérica aunque Pandas la detecte como texto.  
`fecha_registro` debería ser temporal.  
`cliente_id` es numérico técnicamente, pero funciona como identificador.

# 2. Convertir texto a numérico

In [3]:
df["edad"] = pd.to_numeric(df["edad"], errors="coerce")
df.dtypes

,0
cliente_id,int64
edad,int64
ciudad,object
genero,object
fecha_registro,object
ingreso,int64
activo,object


In [4]:
ejemplo = pd.Series(["25", "32", "ERROR", "40"])
pd.to_numeric(ejemplo, errors="coerce")

,0
0,25.0
1,32.0
2,NaN
3,40.0


`errors="coerce"` convierte valores no válidos a `NaN` en lugar de detener la ejecución.

# 3. Convertir fechas

In [5]:
df["fecha_registro"] = pd.to_datetime(df["fecha_registro"], errors="coerce")
df.dtypes

,0
cliente_id,int64
edad,int64
ciudad,object
genero,object
fecha_registro,datetime64[ns]
ingreso,int64
activo,object


# 4. Extraer información temporal

In [16]:
df["anio_registro"] = df["fecha_registro"].dt.year
df["mes_registro"] = df["fecha_registro"].dt.month
df["dia_registro"] = df["fecha_registro"].dt.day
df["dia_semana"] = df["fecha_registro"].dt.day_name()
df["periodo"] = (
    df["anio_registro"].astype(str)
    + "-"
    + df["mes_registro"].astype(str).str.zfill(2)
)
df[["fecha_registro","anio_registro","mes_registro","dia_registro","dia_semana", "periodo"]]


,fecha_registro,anio_registro,mes_registro,dia_registro,dia_semana,periodo
0,2026-01-10,2026,1,10,Saturday,2026-01
1,2026-02-15,2026,2,15,Sunday,2026-02
2,2026-02-20,2026,2,20,Friday,2026-02
3,2026-03-05,2026,3,5,Thursday,2026-03
4,2026-03-21,2026,3,21,Saturday,2026-03


### Pregunta
¿Por qué `mes_registro` o `dia_semana` pueden ser más útiles que conservar solamente la fecha completa?

# 5. Codificación binaria

In [7]:
df["activo_num"] = df["activo"].map({"SI": 1, "NO": 0})
df[["activo", "activo_num"]]

,activo,activo_num
0,SI,1
1,NO,0
2,SI,1
3,SI,1
4,NO,0


In [8]:
df["genero_num"] = df["genero"].map({"M": 0, "F": 1})
df[["genero", "genero_num"]]

,genero,genero_num
0,M,0
1,F,1
2,F,1
3,M,0
4,F,1


Los valores 0 y 1 son códigos. No implican una jerarquía entre categorías.

# 6. One-Hot Encoding

In [17]:
ciudades = pd.get_dummies(df["ciudad"], prefix="ciudad", dtype=int)
ciudades

,ciudad_AREQUIPA,ciudad_CUSCO,ciudad_LIMA
0,0,0,1
1,0,1,0
2,0,0,1
3,1,0,0
4,0,1,0


In [18]:
df = pd.concat([df, ciudades], axis=1)
df.head()

,cliente_id,edad,ciudad,genero,fecha_registro,ingreso,activo,anio_registro,mes_registro,dia_registro,dia_semana,activo_num,genero_num,periodo,ciudad_AREQUIPA,ciudad_CUSCO,ciudad_LIMA
0,101,25,LIMA,M,2026-01-10,1500,SI,2026,1,10,Saturday,1,0,2026-01,0,0,1
1,102,32,CUSCO,F,2026-02-15,3200,NO,2026,2,15,Sunday,0,1,2026-02,0,1,0
2,103,40,LIMA,F,2026-02-20,2100,SI,2026,2,20,Friday,1,1,2026-02,0,0,1
3,104,29,AREQUIPA,M,2026-03-05,1800,SI,2026,3,5,Thursday,1,0,2026-03,1,0,0
4,105,36,CUSCO,F,2026-03-21,4500,NO,2026,3,21,Saturday,0,1,2026-03,0,1,0


Para variables nominales como ciudad, One-Hot Encoding evita crear un orden artificial.

# 7. Variables ordinales

In [19]:
riesgo = pd.Series(["BAJO", "MEDIO", "ALTO", "MEDIO", "BAJO"])

riesgo_num = riesgo.map({
    "BAJO": 1,
    "MEDIO": 2,
    "ALTO": 3
})

pd.DataFrame({"riesgo": riesgo, "riesgo_num": riesgo_num})

,riesgo,riesgo_num
0,BAJO,1
1,MEDIO,2
2,ALTO,3
3,MEDIO,2
4,BAJO,1


Aquí el orden sí tiene sentido: `BAJO < MEDIO < ALTO`.

# 8. Error frecuente: códigos para categorías nominales

In [20]:
df["codigo_ciudad"] = df["ciudad"].astype("category").cat.codes
df[["ciudad","codigo_ciudad"]]

,ciudad,codigo_ciudad
0,LIMA,2
1,CUSCO,1
2,LIMA,2
3,AREQUIPA,0
4,CUSCO,1


Aunque técnicamente funciona, puede inducir un orden que no existe. Para variables nominales es preferible One-Hot Encoding.

# 9. Escalamiento numérico

In [21]:
df[["edad","ingreso"]].describe()

,edad,ingreso
count,5.00000,5.000000
mean,32.40000,2620.000000
std,5.85662,1231.665539
min,25.00000,1500.000000
25%,29.00000,1800.000000
50%,32.00000,2100.000000
75%,36.00000,3200.000000
max,40.00000,4500.000000


# 10. Normalización Min-Max

In [22]:
from sklearn.preprocessing import MinMaxScaler

minmax = MinMaxScaler()
df[["edad_norm","ingreso_norm"]] = minmax.fit_transform(df[["edad","ingreso"]])

df[["edad","edad_norm","ingreso","ingreso_norm"]]

,edad,edad_norm,ingreso,ingreso_norm
0,25,0.000000,1500,0.000000
1,32,0.466667,3200,0.566667
2,40,1.000000,2100,0.200000
3,29,0.266667,1800,0.100000
4,36,0.733333,4500,1.000000


Min-Max lleva normalmente los valores al rango de 0 a 1.

# 11. Estandarización

In [23]:
from sklearn.preprocessing import StandardScaler

standard = StandardScaler()
df[["edad_std","ingreso_std"]] = standard.fit_transform(df[["edad","ingreso"]])

df[["edad","edad_std","ingreso","ingreso_std"]]

,edad,edad_std,ingreso,ingreso_std
0,25,-1.412667,1500,-1.016671
1,32,-0.076360,3200,0.526490
2,40,1.450847,2100,-0.472026
3,29,-0.649063,1800,-0.744348
4,36,0.687243,4500,1.706554


La estandarización deja las variables aproximadamente con media 0 y desviación estándar 1.

# 12. Normalización vs estandarización

In [24]:
pd.DataFrame({
    "Tecnica": ["Min-Max","StandardScaler","Sin escalamiento"],
    "Resultado": [
        "Valores normalmente entre 0 y 1",
        "Media ≈ 0 y desviación estándar ≈ 1",
        "Valores originales"
    ],
    "Uso_general": [
        "Cuando se requiere una escala definida",
        "Algoritmos sensibles a escala",
        "Modelos donde la escala no es crítica"
    ]
})

,Tecnica,Resultado,Uso_general
0,Min-Max,Valores normalmente entre 0 y 1,Cuando se requiere una escala definida
1,StandardScaler,Media ≈ 0 y desviación estándar ≈ 1,Algoritmos sensibles a escala
2,Sin escalamiento,Valores originales,Modelos donde la escala no es crítica


No todos los modelos necesitan escalamiento. Los árboles de decisión, por ejemplo, no dependen de la escala de la misma forma que KNN o K-Means.

# 13. No transformar identificadores

In [25]:
df[["cliente_id"]]

,cliente_id
0,101
1,102
2,103
3,104
4,105


`cliente_id` no debe normalizarse ni interpretarse como una cantidad solo porque sea numérico.

# 14. Construcción del dataset preparado

In [26]:
columnas_modelo = [
    "edad",
    "ingreso",
    "activo_num",
    "ciudad_AREQUIPA",
    "ciudad_CUSCO",
    "ciudad_LIMA"
]

df_preparado = df[columnas_modelo].copy()
df_preparado

,edad,ingreso,activo_num,ciudad_AREQUIPA,ciudad_CUSCO,ciudad_LIMA
0,25,1500,1,0,0,1
1,32,3200,0,0,1,0
2,40,2100,1,0,0,1
3,29,1800,1,1,0,0
4,36,4500,0,0,1,0


# 15. Validación

In [27]:
df_preparado.dtypes

,0
edad,int64
ingreso,int64
activo_num,int64
ciudad_AREQUIPA,int64
ciudad_CUSCO,int64
ciudad_LIMA,int64


In [28]:
df_preparado.isnull().sum()

,0
edad,0
ingreso,0
activo_num,0
ciudad_AREQUIPA,0
ciudad_CUSCO,0
ciudad_LIMA,0


In [29]:
df_preparado.describe()

,edad,ingreso,activo_num,ciudad_AREQUIPA,ciudad_CUSCO,ciudad_LIMA
count,5.00000,5.000000,5.000000,5.000000,5.000000,5.000000
mean,32.40000,2620.000000,0.600000,0.200000,0.400000,0.400000
std,5.85662,1231.665539,0.547723,0.447214,0.547723,0.547723
min,25.00000,1500.000000,0.000000,0.000000,0.000000,0.000000
25%,29.00000,1800.000000,0.000000,0.000000,0.000000,0.000000
50%,32.00000,2100.000000,1.000000,0.000000,0.000000,0.000000
75%,36.00000,3200.000000,1.000000,0.000000,1.000000,1.000000
max,40.00000,4500.000000,1.000000,1.000000,1.000000,1.000000


Después de transformar debemos verificar:
- tipos;
- nulos inesperados;
- codificación;
- escalas;
- columnas generadas.

# 16. Actividad práctica con dataset propio

In [5]:
import pandas as pd
import numpy as np

ruta = "./LI-Small_Trans.csv"
df_proyecto = pd.read_csv(ruta)
print(df_proyecto.shape)
df_proyecto.head()

(6924049, 11)


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:08,11,8000ECA90,11,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,0
1,2022/09/01 00:21,3402,80021DAD0,3402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,0
2,2022/09/01 00:00,11,8000ECA90,1120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0
3,2022/09/01 00:16,3814,8006AD080,3814,8006AD080,12.32,US Dollar,12.32,US Dollar,Reinvestment,0
4,2022/09/01 00:00,20,8006AD530,20,8006AD530,2941.56,US Dollar,2941.56,US Dollar,Reinvestment,0


In [5]:
df_proyecto = df_proyecto.rename(columns={
    "From Bank": "banco_origen",
    "Account": "cuenta_origen",
    "To Bank": "banco_destino",
    "Account.1": "cuenta_destino",
})
df_proyecto.dtypes

,0
Timestamp,object
banco_origen,int64
cuenta_origen,object
banco_destino,int64
cuenta_destino,object
Amount Received,float64
Receiving Currency,object
Amount Paid,float64
Payment Currency,object
Payment Format,object


Identifica como mínimo:
- 1 variable que requiera conversión de tipo;
- 1 variable temporal;
- 2 categóricas;
- 2 numéricas;
- 1 identificador.

In [7]:
filas = [
    ["Timestamp", "object", "datetime64", "pd.to_datetime(format='%Y/%m/%d %H:%M')", "fecha de la transacción, permite análisis temporal"],
    ["banco_origen / banco_destino", "int64", "object", "astype(str)", "código de banco: identificador, no cantidad (demasiados valores para One-Hot)"],
    ["cuenta_origen / cuenta_destino", "object", "object", "sin transformación", "identificador de cuenta, no se normaliza"],
    ["Payment Format", "object", "columnas 0/1", "get_dummies (One-Hot)", "categórica nominal sin orden"],
    ["Receiving Currency", "object", "columnas 0/1", "get_dummies (One-Hot)", "categórica nominal sin orden (15 monedas)"],
    ["Receiving Currency vs Payment Currency", "object", "binaria 0/1", "comparación ==", "indica si hubo cambio de moneda"],
    ["Amount Received / Amount Paid", "float64", "estandarizada", "log1p + StandardScaler", "montos muy asimétricos y en escalas grandes"],
    ["Is Laundering", "int64", "sin cambio", "ninguna", "variable objetivo"],
]

plan_transformacion = pd.DataFrame(filas, columns=["Variable", "Tipo_actual", "Tipo_esperado", "Transformacion", "Justificacion"])
plan_transformacion

,Variable,Tipo_actual,Tipo_esperado,Transformacion,Justificacion
0,Timestamp,object,datetime64,pd.to_datetime(format='%Y/%m/%d %H:%M'),"fecha de la transacción, permite análisis temp..."
1,banco_origen / banco_destino,int64,object,astype(str),"código de banco: identificador, no cantidad (d..."
2,cuenta_origen / cuenta_destino,object,object,sin transformación,"identificador de cuenta, no se normaliza"
3,Payment Format,object,columnas 0/1,get_dummies (One-Hot),categórica nominal sin orden
4,Receiving Currency,object,columnas 0/1,get_dummies (One-Hot),categórica nominal sin orden (15 monedas)
5,Receiving Currency vs Payment Currency,object,binaria 0/1,comparación ==,indica si hubo cambio de moneda
6,Amount Received / Amount Paid,float64,estandarizada,log1p + StandardScaler,montos muy asimétricos y en escalas grandes
7,Is Laundering,int64,sin cambio,ninguna,variable objetivo


Aplica como mínimo 4 transformaciones:
1. conversión de tipo;
2. transformación temporal;
3. codificación categórica;
4. escalamiento numérico.

In [6]:
# Convierte la columna Timestamp a formato datetime
df_proyecto["Timestamp"] = pd.to_datetime(
    df_proyecto["Timestamp"], errors="coerce"
)

# Verifica los tipos de datos actualizados
df_proyecto.dtypes

Timestamp             datetime64[ns]
From Bank                      int64
Account                       object
To Bank                        int64
Account.1                     object
Amount Received              float64
Receiving Currency            object
Amount Paid                  float64
Payment Currency              object
Payment Format                object
Is Laundering                  int64
dtype: object

In [18]:
import pandas as pd

# 1. Asegúrate de que la columna de fecha esté en formato datetime dentro de df_proyecto
df_proyecto["Timestamp"] = pd.to_datetime(df_proyecto["Timestamp"])

# 2. Extrae la información temporal usando df_proyecto
df_proyecto["anio_registro"] = df_proyecto["Timestamp"].dt.year
df_proyecto["mes_registro"] = df_proyecto["Timestamp"].dt.month
df_proyecto["dia_registro"] = df_proyecto["Timestamp"].dt.day
df_proyecto["dia_semana"] = df_proyecto["Timestamp"].dt.day_name()

# 3. Verifica el resultado con las columnas que necesites
df_proyecto[
    [
        "Timestamp",
        "anio_registro",
        "mes_registro",
        "dia_registro",
        "dia_semana",
    ]
].head()

,Timestamp,anio_registro,mes_registro,dia_registro,dia_semana
0,2022-09-01 00:08:00,2022,9,1,Thursday
1,2022-09-01 00:21:00,2022,9,1,Thursday
2,2022-09-01 00:00:00,2022,9,1,Thursday
3,2022-09-01 00:16:00,2022,9,1,Thursday
4,2022-09-01 00:00:00,2022,9,1,Thursday


In [20]:
# Cuenta cuántas veces aparece cada valor único en la columna Is Laundering
df_proyecto["Is Laundering"].value_counts()

Is Laundering
0    6920484
1       3565
Name: count, dtype: int64

In [7]:
# Muestra el resumen estadístico de las columnas de montos
df_proyecto[["Amount Received", "Amount Paid"]].describe()

,Amount Received,Amount Paid
count,6.924049e+06,6.924049e+06
mean,6.324067e+06,4.676036e+06
std,2.105371e+09,1.544099e+09
min,1.000000e-06,1.000000e-06
25%,1.742100e+02,1.753800e+02
50%,1.397620e+03,1.399440e+03
75%,1.229633e+04,1.222687e+04
max,3.644854e+12,3.644854e+12


In [8]:
from sklearn.preprocessing import MinMaxScaler

# 1. Crear la instancia del MinMaxScaler
minmax = MinMaxScaler()

# 2. Aplicar la transformación y guardar en nuevas columnas
df_proyecto[["Amount_Received_norm", "Amount_Paid_norm"]] = (
    minmax.fit_transform(df_proyecto[["Amount Received", "Amount Paid"]])
)

# 3. Visualizar el resultado comparando las columnas originales con las normalizadas
df_proyecto[
    [
        "Amount Received",
        "Amount_Received_norm",
        "Amount Paid",
        "Amount_Paid_norm",
    ]
].head()

,Amount Received,Amount_Received_norm,Amount Paid,Amount_Paid_norm
0,3195403.00,8.766890e-07,3195403.00,8.766890e-07
1,1858.96,5.100232e-10,1858.96,5.100232e-10
2,592571.00,1.625774e-07,592571.00,1.625774e-07
3,12.32,3.380108e-12,12.32,3.380108e-12
4,2941.56,8.070447e-10,2941.56,8.070447e-10


In [9]:
from sklearn.preprocessing import MinMaxScaler

# 1. Configurar Pandas para mostrar decimales normales en lugar de notación científica
pd.options.display.float_format = "{:.6f}".format

# 2. Crear la instancia del MinMaxScaler
minmax = MinMaxScaler()

# 3. Aplicar la transformación y guardar en las nuevas columnas
df_proyecto[["Amount_Received_norm", "Amount_Paid_norm"]] = (
    minmax.fit_transform(df_proyecto[["Amount Received", "Amount Paid"]])
)

# 4. Visualizar el resultado igual que en el ejemplo
df_proyecto[
    [
        "Amount Received",
        "Amount_Received_norm",
        "Amount Paid",
        "Amount_Paid_norm",
    ]
].head()

,Amount Received,Amount_Received_norm,Amount Paid,Amount_Paid_norm
0,3195403.000000,0.000001,3195403.000000,0.000001
1,1858.960000,0.000000,1858.960000,0.000000
2,592571.000000,0.000000,592571.000000,0.000000
3,12.320000,0.000000,12.320000,0.000000
4,2941.560000,0.000000,2941.560000,0.000000


# 17. Matriz final de transformación

In [7]:
filas = [
    ["Timestamp", "object (texto)", "pd.to_datetime + hora y dia_semana", "datetime64 + hora (0-23)", "permite análisis por hora y día"],
    ["banco_origen / banco_destino", "int64", "astype(str)", "object (identificador)", "código, no cantidad; demasiados valores para One-Hot"],
    ["cuenta_origen / cuenta_destino", "object", "ninguna", "object (identificador)", "no se normaliza ni codifica"],
    ["Payment Format", "object (7 valores)", "One-Hot (get_dummies)", "7 columnas 0/1 formato_*", "variable nominal sin orden"],
    ["Receiving Currency", "object (15 valores)", "One-Hot (get_dummies)", "15 columnas 0/1 moneda_*", "variable nominal sin orden"],
    ["Receiving Currency vs Payment Currency", "object", "comparación", "misma_moneda 0/1", "indica cambio de moneda"],
    ["Amount Received / Amount Paid", "float64 (escala muy amplia)", "log1p + StandardScaler", "monto_recibido_std / monto_pagado_std", "reduce asimetría y diferencia de escala"],
    ["Is Laundering", "int64", "ninguna", "int64 (0/1)", "variable objetivo; muy desbalanceada (~0.05% positivos)"],
]

matriz_transformacion = pd.DataFrame(filas, columns=["Variable", "Estado_original", "Transformacion_aplicada", "Estado_final", "Justificacion"])
matriz_transformacion

,Variable,Estado_original,Transformacion_aplicada,Estado_final,Justificacion
0,Timestamp,object (texto),pd.to_datetime + hora y dia_semana,datetime64 + hora (0-23),permite análisis por hora y día
1,banco_origen / banco_destino,int64,astype(str),object (identificador),"código, no cantidad; demasiados valores para O..."
2,cuenta_origen / cuenta_destino,object,ninguna,object (identificador),no se normaliza ni codifica
3,Payment Format,object (7 valores),One-Hot (get_dummies),7 columnas 0/1 formato_*,variable nominal sin orden
4,Receiving Currency,object (15 valores),One-Hot (get_dummies),15 columnas 0/1 moneda_*,variable nominal sin orden
5,Receiving Currency vs Payment Currency,object,comparación,misma_moneda 0/1,indica cambio de moneda
6,Amount Received / Amount Paid,float64 (escala muy amplia),log1p + StandardScaler,monto_recibido_std / monto_pagado_std,reduce asimetría y diferencia de escala
7,Is Laundering,int64,ninguna,int64 (0/1),variable objetivo; muy desbalanceada (~0.05% p...


In [6]:
# 1. Conversión de tipo: códigos de banco son identificadores
df_proyecto[["banco_origen", "banco_destino"]] = df_proyecto[["banco_origen", "banco_destino"]].astype(str)

# 2. Transformación temporal
df_proyecto["fecha"] = pd.to_datetime(df_proyecto["Timestamp"], format="%Y/%m/%d %H:%M", errors="coerce")
df_proyecto["hora"] = df_proyecto["fecha"].dt.hour
df_proyecto["dia_semana"] = df_proyecto["fecha"].dt.day_name()

# 3. Codificación categórica
df_proyecto["misma_moneda"] = (df_proyecto["Receiving Currency"] == df_proyecto["Payment Currency"]).astype(int)
formatos = pd.get_dummies(df_proyecto["Payment Format"], prefix="formato", dtype=int)
monedas = pd.get_dummies(df_proyecto["Receiving Currency"], prefix="moneda", dtype=int)
df_proyecto = pd.concat([df_proyecto, formatos, monedas], axis=1)

# 4. Escalamiento numérico (log1p por asimetría de montos, luego StandardScaler)
from sklearn.preprocessing import StandardScaler

montos = ["Amount Received", "Amount Paid"]
df_proyecto[["monto_recibido_log", "monto_pagado_log"]] = np.log1p(df_proyecto[montos])
escala = StandardScaler()
df_proyecto[["monto_recibido_std", "monto_pagado_std"]] = escala.fit_transform(
    df_proyecto[["monto_recibido_log", "monto_pagado_log"]]
)

# Dataset preparado y validación
df_preparado_proyecto = df_proyecto[
    ["hora", "misma_moneda", "monto_recibido_std", "monto_pagado_std"]
    + list(formatos.columns) + list(monedas.columns) + ["Is Laundering"]
].copy()
print(df_preparado_proyecto.dtypes)
print(df_preparado_proyecto.isnull().sum())
df_preparado_proyecto.describe()

hora                          int32
misma_moneda                  int64
monto_recibido_std          float64
monto_pagado_std            float64
formato_ACH                   int64
formato_Bitcoin               int64
formato_Cash                  int64
formato_Cheque                int64
formato_Credit Card           int64
formato_Reinvestment          int64
formato_Wire                  int64
moneda_Australian Dollar      int64
moneda_Bitcoin                int64
moneda_Brazil Real            int64
moneda_Canadian Dollar        int64
moneda_Euro                   int64
moneda_Mexican Peso           int64
moneda_Ruble                  int64
moneda_Rupee                  int64
moneda_Saudi Riyal            int64
moneda_Shekel                 int64
moneda_Swiss Franc            int64
moneda_UK Pound               int64
moneda_US Dollar              int64
moneda_Yen                    int64
moneda_Yuan                   int64
Is Laundering                 int64
dtype: object
hora          

,hora,misma_moneda,monto_recibido_std,monto_pagado_std,formato_ACH,formato_Bitcoin,formato_Cash,formato_Cheque,formato_Credit Card,formato_Reinvestment,...,moneda_Ruble,moneda_Rupee,moneda_Saudi Riyal,moneda_Shekel,moneda_Swiss Franc,moneda_UK Pound,moneda_US Dollar,moneda_Yen,moneda_Yuan,Is Laundering
count,6.924049e+06,6.924049e+06,6.924049e+06,6.924049e+06,6.924049e+06,6.924049e+06,6.924049e+06,6.924049e+06,6.924049e+06,6.924049e+06,...,6.924049e+06,6.924049e+06,6.924049e+06,6.924049e+06,6.924049e+06,6.924049e+06,6.924049e+06,6.924049e+06,6.924049e+06,6.924049e+06
mean,1.050052e+01,9.857199e-01,1.309803e-15,1.244397e-15,1.150455e-01,4.465711e-02,9.469719e-02,3.615165e-01,2.571312e-01,9.394185e-02,...,1.287397e-02,4.971614e-02,3.782209e-02,2.560612e-02,2.023036e-02,1.415357e-02,3.664391e-01,3.056463e-02,6.859830e-02,5.148722e-04
std,7.370397e+00,1.186430e-01,1.000000e+00,1.000000e+00,3.190769e-01,2.065499e-01,2.927963e-01,4.804398e-01,4.370524e-01,2.917478e-01,...,1.127308e-01,2.173579e-01,1.907658e-01,1.579571e-01,1.407874e-01,1.181239e-01,4.818314e-01,1.721349e-01,2.527698e-01,2.268495e-02
min,0.000000e+00,0.000000e+00,-2.124014e+00,-2.132445e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,4.000000e+00,1.000000e+00,-6.298756e-01,-6.296907e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,1.000000e+01,1.000000e+00,-2.907849e-02,-2.776231e-02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,1.700000e+01,1.000000e+00,5.996695e-01,6.017742e-01,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,2.300000e+01,1.000000e+00,6.241665e+00,6.270649e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,...,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00


### Ejemplo de estructura

| Variable | Estado original | Transformación | Estado final | Justificación |
|---|---|---|---|---|
| edad | object | to_numeric() | numérica | representa años |
| fecha | object | to_datetime() | datetime | análisis temporal |
| ciudad | texto | One-Hot | columnas 0/1 | variable nominal |
| activo | SI/NO | map | 0/1 | variable binaria |
| ingreso | escala original | StandardScaler | estandarizada | evitar diferencia de escala |

# 18. Preguntas de cierre

1. ¿Por qué un identificador no debería normalizarse?
   Porque es una clave primaria
3. ¿Cuándo utilizar One-Hot Encoding?
   Transformar variables categóricas nominales en un formato numérico que los algoritmos de Machine Learning puedan interpretar correctamente.
5. ¿Qué problema existe al codificar Lima=1, Cusco=2 y Arequipa=3?
   Puede introducir un orden que no existe
7. ¿Qué diferencia existe entre una variable nominal y una ordinal?
   La ordinal tiene un orden o jerarquia mientras que la nominal no, pese a ambas ser categoricas
9. ¿Qué información puede extraerse de una fecha?
   Componentes del calendario, día de la semana, indicadores como fin de semana o feriado, y tiempo transcurrido.
11. ¿Normalizar siempre mejora un dataset?
    No, en modelos basados en árboles o cuando las características ya comparten la misma escala no aporta beneficios.
13. ¿Cuál es la diferencia entre normalización y estandarización?
    La normalización escala los valores a un rango fijo como 0 a 1, la estandarización ajusta los datos a media 0 y desviación estándar 1.
15. ¿Por qué debemos conservar el dataset original?
    Para asegurar la reproducibilidad, auditoría, corregir errores en las transformaciones y probar nuevos enfoques sin perder la fuente.
17. ¿Qué riesgo existe al transformar variables sin documentar?
    Perder la trazabilidad, causar fuga de datos, dificultar la interpretación y fallar al desplegar el modelo en producción.
19. ¿Qué validaciones deben realizarse después de transformar?
    Revisar ausencia de nulos y duplicados, validar las dimensiones del dataset y comprobar que los rangos o escalas sean los esperados.

# 19. Producto final

Cada grupo debe entregar:
- dataset original;
- dataset preparado;
- plan de transformación;
- matriz final de transformación;
- conclusión de máximo 5 líneas.

> **Conclusión:** ¿Qué transformaciones fueron necesarias y por qué mejoran la utilidad del dataset?
> Al transformar las fechas en datos temporales, pudimos ordenar la información y extraer con facilidad detalles clave como el año, mes y día. Además, al normalizar los montos con el escalamiento y mantener intactos los códigos de identificación, logramos dejar el dataset totalmente preparado y listo para cualquier análisis futuro.

# 20. Cierre

### 20. Cierre – Resultados obtenidos

En esta sesión trabajamos con un dataset de **6 924 049 transacciones financieras** y 11 columnas originales.

1. Convertimos las fechas y extrajimos información temporal, como la hora y el día de la semana.
2. Aplicamos One-Hot Encoding a 7 formatos de pago y 15 tipos de moneda, además de estandarizar los montos.
3. Obtuvimos un dataset preparado con **27 columnas y sin valores nulos**.
4. Encontramos **3 565 transacciones etiquetadas como lavado de dinero**, equivalentes aproximadamente al 0,0515 % del total.
5. Concluimos que las transformaciones facilitaron el análisis de los datos, aunque existe un fuerte desbalance entre las transacciones normales y las de lavado de dinero.